# Test Accuracy: Baseline ResNet-18 vs N2UQ Quantised vs TLMAC LUT-Based Inference

Compares three inference paths on the same inputs:
1. **Baseline** — standard torchvision ResNet-18 (float32 weights)
2. **N2UQ** — Nonuniform-to-Uniform Quantised ResNet-18 (3-bit, from `.pth.tar`)
3. **TLMAC** — emulated TLMAC: loads LUT INIT hex files and performs lookup-based
   convolution to replicate what the FPGA PE does in hardware

All three use identical inputs. TLMAC emulation uses the exported hex files from
`tlmac_compile.ipynb` to reconstruct the convolution results bit-serially.

In [ ]:
!apt-get install -y git-lfs > /dev/null 2>&1 && git lfs install
!git clone https://github.com/leozqi/resnet18_n2uq.git || true
%cd resnet18_n2uq
!pip install torch torchvision numpy matplotlib 2>/dev/null

# Unpack TLMAC hex files if present
import os, tarfile
if os.path.exists('tlmac_output_new.tar.gz'):
    with tarfile.open('tlmac_output_new.tar.gz', 'r:gz') as tar:
        tar.extractall('.')
    print('Extracted tlmac_output_new.tar.gz')
elif os.path.exists('tlmac_output.tar.gz'):
    with tarfile.open('tlmac_output.tar.gz', 'r:gz') as tar:
        tar.extractall('.')
    print('Extracted tlmac_output.tar.gz')


In [ ]:
import math, os, json
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torchvision.models as vision_models
from torchvision import datasets, transforms

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Model Definitions

In [ ]:
# --- N2UQ architecture (from resnet.py) ---
N_BIT = 3

class LearnableBias(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(1, ch, 1, 1))
    def forward(self, x):
        return x + self.bias.expand_as(x)

class LTQ(nn.Module):
    def __init__(self, num_bits):
        super().__init__()
        self.n_val = 2 ** num_bits - 1
        self.interval = 2.0 / self.n_val
        self.start = nn.Parameter(torch.tensor([0.0]))
        self.a = nn.Parameter(torch.tensor([self.interval] * self.n_val))
        self.scale1 = nn.Parameter(torch.tensor([1.0]))
        self.scale2 = nn.Parameter(torch.tensor([1.0]))

    def forward(self, x):
        x = x * self.scale1
        a_pos = torch.where(self.a > 1e-3, self.a, torch.tensor(1e-3))
        step_right = torch.tensor(0.0)
        x_f = x; x_b = x
        for i in range(self.n_val):
            step_right = step_right + self.interval
            if i == 0:
                th_f = self.start + a_pos[0] / 2
                th_b = self.start
                x_f = torch.where(x > th_f, step_right, torch.tensor(0.0))
                x_b = torch.where(x > th_b, self.interval / a_pos[i] * (x - th_b) + step_right - self.interval, torch.tensor(0.0))
            else:
                th_f = th_f + a_pos[i - 1] / 2 + a_pos[i] / 2
                th_b = th_b + a_pos[i - 1]
                x_f = torch.where(x > th_f, step_right, x_f)
                x_b = torch.where(x > th_b, self.interval / a_pos[i] * (x - th_b) + step_right - self.interval, x_b)
        th_b = th_b + a_pos[-1]
        x_b = torch.where(x > th_b, torch.tensor(2.0), x_b)
        return (x_f.detach() + x_b - x_b.detach()) * self.scale2

class HardQuantizeConv(nn.Module):
    def __init__(self, in_ch, out_ch, num_bits, ks=3, stride=1, padding=1):
        super().__init__()
        self.stride = stride; self.padding = padding
        self.num_bits = num_bits; self.kernel_size = ks
        self.clip_val = nn.Parameter(torch.tensor([2.0]))
        self.shape = (out_ch, in_ch, ks, ks)
        self.weight = nn.Parameter((torch.rand(self.shape) - 0.5) * 0.001)

    def quantise_fwd(self):
        rw = self.weight
        gamma = (2**self.num_bits - 1) / (2**(self.num_bits - 1))
        sf = gamma * rw.abs().mean(dim=[1, 2, 3], keepdim=True).detach()
        sw = rw / sf
        cw = sw.clamp(-self.clip_val / 2, self.clip_val / 2)
        n = float(2 ** self.num_bits - 1) / self.clip_val
        return sf * (torch.round((cw + self.clip_val / 2) * n) / n - self.clip_val / 2)

    def forward(self, x):
        return F.conv2d(x, self.quantise_fwd(), stride=self.stride, padding=self.padding)

    def get_integer_weights(self):
        with torch.no_grad():
            rw = self.weight
            gamma = (2**self.num_bits - 1) / (2**(self.num_bits - 1))
            sf = gamma * rw.abs().mean(dim=[1, 2, 3], keepdim=True)
            sw = rw / sf
            cw = sw.clamp(-self.clip_val / 2, self.clip_val / 2)
            n = float(2 ** self.num_bits - 1) / self.clip_val
            return torch.round((cw + self.clip_val / 2) * n).to(torch.int8)  # 0..2^n-1

    def get_quantised_real_weights(self):
        return self.quantise_fwd().detach()

In [ ]:
class BasicBlockQ(nn.Module):
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.bias11 = LearnableBias(inplanes)
        self.prelu1 = nn.PReLU(inplanes)
        self.bias12 = LearnableBias(inplanes)
        self.quan1 = LTQ(N_BIT)
        self.conv1 = HardQuantizeConv(inplanes, planes, N_BIT, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(planes)
        self.bias21 = LearnableBias(planes)
        self.prelu2 = nn.PReLU(planes)
        self.bias22 = LearnableBias(planes)
        self.quan2 = LTQ(N_BIT)
        self.conv2 = HardQuantizeConv(planes, planes, N_BIT, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.bias31 = LearnableBias(planes)
        self.prelu3 = nn.PReLU(planes)
        self.bias32 = LearnableBias(planes)
    def forward(self, x):
        identity = x
        out = self.bias12(self.prelu1(self.bias11(x)))
        out = self.bn1(self.conv1(self.quan1(out)))
        out = self.bias22(self.prelu2(self.bias21(out)))
        out = self.bn2(self.conv2(self.quan2(out)))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return self.bias32(self.prelu3(self.bias31(out)))

class N2UQ_ResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        self.layer1 = self._blk(64, 64, 2)
        self.layer2 = self._blk(64, 128, 2, stride=2)
        self.layer3 = self._blk(128, 256, 2, stride=2)
        self.layer4 = self._blk(256, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, 1000)
    def _blk(self, inp, planes, blocks, stride=1):
        ds = None
        if stride != 1 or inp != planes:
            ds = nn.Sequential(LTQ(N_BIT), HardQuantizeConv(inp, planes, N_BIT, 1, stride, 0), nn.BatchNorm2d(planes))
        layers = [BasicBlockQ(inp, planes, stride, ds)]
        for _ in range(1, blocks): layers.append(BasicBlockQ(planes, planes))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(torch.flatten(self.avgpool(x), 1))

In [ ]:
# --- N2UQ model ---
model_n2uq = N2UQ_ResNet18().to(device)
n2uq_ckpt = torch.load('pretrained_models/quantize_downsampling_true/real_res18.pth.tar', map_location='cpu')
if isinstance(n2uq_ckpt, dict) and 'state_dict' in n2uq_ckpt:
    n2uq_ckpt = n2uq_ckpt['state_dict']
model_n2uq.load_state_dict(n2uq_ckpt, strict=False)
model_n2uq.eval()
for p in model_n2uq.parameters(): p.requires_grad_(False)

# --- Baseline model ---
model_bl = vision_models.resnet18(weights=None)
model_bl.load_state_dict(torch.load('pretrained_models/quantize_downsampling_false/resnet18-f37072fd(pytorch_model).pth', map_location='cpu'))
model_bl = model_bl.to(device).eval()
for p in model_bl.parameters(): p.requires_grad_(False)

print(f'N2UQ ResNet-18:   {sum(p.numel() for p in model_n2uq.parameters()):,} params')
print(f'Baseline ResNet-18: {sum(p.numel() for p in model_bl.parameters()):,} params')

In [ ]:
x_test = torch.randn(8, 3, 224, 224, device=device)
with torch.no_grad():
    out_bl = model_bl(x_test)
    out_n2uq = model_n2uq(x_test)
print(f'Baseline shape: {out_bl.shape}')
print(f'N2UQ shape:     {out_n2uq.shape}')
print(f'Baseline top-1: {out_bl.argmax(1).tolist()}')
print(f'N2UQ top-1:     {out_n2uq.argmax(1).tolist()}')

## 2. TLMAC Software Emulator

Reproduces what the TLMAC PE hardware does in software:
for each bit-serial iteration, feed one activation bit into the LUT truth table,
read out the partial MAC result, shift by bit index, and accumulate.

This lets us compare the TLMAC output against the original PyTorch computation.

In [ ]:
class TLMACConvEmulator:
    """Bit-serial TLMAC PE emulator.

    Loads hex files from tlmac_output/ and performs the exact same
    LUT-based convolution the FPGA would execute: bit-serial over
    activation bits, LUT-6 lookup, switch-network routing, partial-sum
    accumulation.
    """

    def __init__(self, layer_dir, module, meta):
        self.module = module
        D_o, D_i, Dk, _ = module.shape
        self.D_o = D_o
        self.D_i = D_i
        self.DK = Dk
        s = module.stride
        self.stride = s if isinstance(s, int) else s[0]
        p = module.padding
        self.padding = p if isinstance(p, int) else p[0]
        self.D_s = meta['D_s']
        self.D_p = meta['D_p']  # 64 * DK
        self.n_arr = meta['n_arr']
        self.NLUT = meta['NLUT_PER_ARR']
        self.NCLUS = meta['NCLUS']
        self.G = meta['G']
        self.BW = meta['BW']
        self.BA = meta.get('BA', meta['BW'])
        self.o_blocks = D_o // 64

        # cluster_map: step -> wg_sel
        self.cluster_map = []
        with open(os.path.join(layer_dir, 'cluster_map.hex')) as f:
            for line in f:
                self.cluster_map.append(int(line.strip(), 16))

        # LUT INIT: [n_arr, NLUT] int64
        self.lut_init = torch.zeros(self.n_arr, self.NLUT, dtype=torch.int64)
        for ai in range(self.n_arr):
            with open(os.path.join(layer_dir, f'lut_arr{ai}.hex')) as f:
                for k, line in enumerate(f):
                    self.lut_init[ai, k] = int(line.strip(), 16)

        # Precompute lut_table[ai, addr] = NLUT-bit signed result
        # addr = {wg_sel[2:0], act_bits[2:0]} = 6 bits (64 entries)
        addr_range = torch.arange(64)
        table = torch.zeros(self.n_arr, 64, dtype=torch.int64)
        for k in range(self.NLUT):
            table |= ((self.lut_init[:, k].unsqueeze(1) >> addr_range) & 1) << k
        # Sign-extend
        sign_bit = 1 << (self.NLUT - 1)
        self.lut_table = torch.where(table >= sign_bit, table - (1 << self.NLUT), table)  # [n_arr, 64]

        # routing_bitmap -> route_map[p] = arr_index
        with open(os.path.join(layer_dir, 'routing_bitmap.hex')) as f:
            words = [int(line.strip(), 16) for line in f]
        rb_flat = 0
        for wi, w in enumerate(words):
            rb_flat |= w << (wi * 64)
        self.route_map = torch.zeros(self.D_p, dtype=torch.int64)
        for p in range(self.D_p):
            for ai in range(self.n_arr):
                if (rb_flat >> (p * self.n_arr + ai)) & 1:
                    self.route_map[p] = ai
                    break

    def forward(self, x):
        """TLMAC bit-serial convolution.
        x: [batch, D_i, H, W] float activations.
        Returns: [batch, D_o, oH, oW] float output (integer partial sums).
        """
        B, _, H, W = x.shape
        stride, pad, DK = self.stride, self.padding, self.DK
        oH = (H + 2 * pad - DK) // stride + 1
        oW = (W + 2 * pad - DK) // stride + 1
        x_pad = F.pad(x, [pad, pad, pad, pad])  # [B, D_i, H+2p, W+2p]

        # Quantize activations to BA-bit unsigned [0, 2^BA - 1]
        clip = float(self.module.clip_val)
        n_q = float(2 ** self.BA - 1) / clip
        x_q = torch.round((x_pad.clamp(-clip / 2, clip / 2) + clip / 2) * n_q)
        x_q = x_q.clamp(0, 2 ** self.BA - 1).to(torch.int64)  # [B, D_i, H+2p, W+2p]

        # Precompute spatial indices for all (oh, ow) positions
        oh_idx = torch.arange(oH).unsqueeze(1).expand(-1, oW).reshape(-1)  # [oH*oW]
        ow_idx = torch.arange(oW).unsqueeze(0).expand(oH, -1).reshape(-1)  # [oH*oW]
        N_spatial = oH * oW

        output = torch.zeros(B, self.D_o, oH, oW, dtype=torch.float32)

        # LUT lookup cache: lut_table[self.route_map, addr] -> [D_p] for each addr
        # Precompute: routed_table[addr, p] = lut_table[route_map[p], addr]
        routed_table = self.lut_table[self.route_map]  # [D_p, 64]

        for b_idx in range(B):
            psum = torch.zeros(N_spatial, self.D_o, dtype=torch.int64)

            for s in range(self.D_s):
                ic = s // self.o_blocks
                o_blk = s % self.o_blocks
                wg_sel = self.cluster_map[s]

                for kr in range(DK):
                    # Activation values for all spatial positions: [N_spatial, DK]
                    h_pos = oh_idx * stride + kr  # [N_spatial]
                    w_start = ow_idx * stride      # [N_spatial]
                    w_offsets = torch.arange(DK).unsqueeze(0)  # [1, DK]
                    w_idx = (w_start.unsqueeze(1) + w_offsets).clamp(0, x_q.shape[3] - 1)  # [N, DK]
                    h_idx = h_pos.unsqueeze(1).expand(-1, DK)  # [N, DK]
                    act_vals = x_q[b_idx, ic, h_idx, w_idx]  # [N_spatial, DK]

                    for bit_idx in range(self.BA):
                        # Extract bit bit_idx: [N_spatial, DK]
                        act_bits = (act_vals >> bit_idx) & 1  # [N, DK]
                        # Compute address for each spatial position: [N]
                        act_val = act_bits[:, 0] | (act_bits[:, 1] << 1) | (act_bits[:, 2] << 2)  # [N]
                        addr = (wg_sel << self.G) | act_val  # [N]

                        # LUT lookup + route: [D_p, N] -> [N, D_p]
                        mac_results = routed_table[:, addr].T  # [N, D_p]

                        # Map D_p outputs to D_o channels
                        # p = oc * DK + kr, out_ch = o_blk*64 + oc
                        oc_idx = torch.arange(64)
                        p_idx = oc_idx * DK + kr  # [64]
                        out_ch = o_blk * 64 + oc_idx  # [64]
                        valid = (out_ch < self.D_o) & (p_idx < self.D_p)
                        vals = mac_results[:, p_idx[valid]]  # [N, valid_count]
                        psum[:, out_ch[valid]] += vals << bit_idx

            output[b_idx] = psum.reshape(oH, oW, self.D_o).permute(2, 0, 1)

        return output

print('TLMACConvEmulator class defined (bit-serial, vectorised spatial)')


In [ ]:
TLMAC_DIR = 'tlmac_output'
has_tlmac = os.path.isdir(os.path.join(TLMAC_DIR, 'weights'))

tlmac_emulators = {}  # layer_name -> TLMACConvEmulator

if has_tlmac:
    meta_path = os.path.join(TLMAC_DIR, 'metadata.json')
    with open(meta_path) as f:
        tlmac_meta = json.load(f)
    print(f'TLMAC output found: {len(tlmac_meta)} layers')
    for k, v in tlmac_meta.items():
        print(f'  {k}: {v["n_arr"]} arrays, {v["clusters"]} clusters')

    # Instantiate emulators for each 3x3 conv layer
    for name, module in model_n2uq.named_modules():
        if isinstance(module, HardQuantizeConv) and module.kernel_size in (3, (3, 3)):
            safe = name.replace('.', '_')
            if safe in tlmac_meta:
                layer_dir = os.path.join(TLMAC_DIR, 'weights', safe)
                tlmac_emulators[name] = TLMACConvEmulator(layer_dir, module, tlmac_meta[safe])
    print(f'Instantiated {len(tlmac_emulators)} TLMAC emulators')
else:
    print(f'TLMAC output not found at {TLMAC_DIR}/')
    print('Run tlmac_compile.ipynb first.')


## 3. Inference Comparison on Synthetic Data

In [ ]:
N_SAMPLES = 32
x_eval = torch.randn(N_SAMPLES, 3, 224, 224, device=device)

with torch.no_grad():
    logits_bl = model_bl(x_eval)
    logits_n2uq = model_n2uq(x_eval)

# Top-k comparison
top1_bl = logits_bl.argmax(1)
top1_n2uq = logits_n2uq.argmax(1)
top5_bl = logits_bl.topk(5, dim=1).indices
top5_n2uq = logits_n2uq.topk(5, dim=1).indices

agreement_1 = (top1_bl == top1_n2uq).float().mean().item()
# top-5 agreement: does baseline top-1 appear in N2UQ top-5?
top1_in_top5 = torch.tensor([(top1_bl[i] in top5_n2uq[i]) for i in range(N_SAMPLES)]).float().mean().item()

print(f'Samples: {N_SAMPLES}')
print(f'Top-1 agreement (BL vs N2UQ):  {agreement_1:.1%}')
print(f'BL top-1 in N2UQ top-5:        {top1_in_top5:.1%}')

# Logit distance (L2 norm, normalised)
logit_dist = (logits_bl - logits_n2uq).norm(dim=1).mean()
print(f'Mean L2 logit distance:         {logit_dist:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Top-1 confidence
conf_bl = logits_bl.softmax(1).max(1).values.cpu().numpy()
conf_n2uq = logits_n2uq.softmax(1).max(1).values.cpu().numpy()
axes[0].hist(conf_bl, bins=20, alpha=0.5, label='Baseline')
axes[0].hist(conf_n2uq, bins=20, alpha=0.5, label='N2UQ')
axes[0].set_title('Top-1 Confidence')
axes[0].legend()

# Logit L2 distance per sample
per_sample_dist = (logits_bl - logits_n2uq).norm(dim=1).cpu().numpy()
axes[1].bar(range(N_SAMPLES), per_sample_dist)
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('L2 distance')
axes[1].set_title('Per-sample logit distance')

# Top-1 class agreement scatter
axes[2].scatter(logits_bl.max(1).values.cpu().numpy(),
                logits_n2uq.max(1).values.cpu().numpy(),
                alpha=0.5, s=20)
mn = min(logits_bl.max(1).values.min().item(), logits_n2uq.max(1).values.min().item())
mx = max(logits_bl.max(1).values.max().item(), logits_n2uq.max(1).values.max().item())
axes[2].plot([mn, mx], [mn, mx], 'r--', alpha=0.5)
axes[2].set_xlabel('Baseline max logit')
axes[2].set_ylabel('N2UQ max logit')
axes[2].set_title('Max logit comparison')

plt.tight_layout()
plt.show()

## 4. TLMAC Layer-by-Layer Verification

For each exported layer, verify that the LUT INIT values reproduce
the same convolution result as the N2UQ quantised weights.
This confirms the hex files are correct before bitstream generation.

In [ ]:
# Find all HardQuantizeConv modules
n2uq_conv_layers = []
for name, module in model_n2uq.named_modules():
    if isinstance(module, HardQuantizeConv) and module.kernel_size in (3, (3, 3)):
        n2uq_conv_layers.append((name, module))

print(f'Verifying {len(n2uq_conv_layers)} 3x3 conv layers...')
print()

verification_results = []

for name, module in n2uq_conv_layers:
    # Use the correct input channel count for this layer
    D_o, D_i, _, _ = module.shape
    x_ver = torch.randn(1, D_i, 8, 8, device=device)

    # N2UQ quantised forward
    with torch.no_grad():
        out_n2uq = module(x_ver)

    if has_tlmac and name in tlmac_emulators:
        emu = tlmac_emulators[name]
        with torch.no_grad():
            out_tlmac = emu.forward(x_ver)

        t_min = out_tlmac.min().item()
        t_max = out_tlmac.max().item()
        t_mean = out_tlmac.mean().item()
        n_min = out_n2uq.min().item()
        n_max = out_n2uq.max().item()
        n_mean = out_n2uq.mean().item()

        flat_t = out_tlmac.flatten().float()
        flat_n = out_n2uq.flatten().float()
        if flat_t.std() > 1e-8 and flat_n.std() > 1e-8:
            corr = torch.corrcoef(torch.stack([flat_t, flat_n]))[0, 1].item()
        else:
            corr = float('nan')

        short = name.replace('.conv', '.c')
        print(f'  {short}:')
        print(f'    N2UQ:  [{n_min:.3f}, {n_max:.3f}], mean={n_mean:.3f}')
        print(f'    TLMAC: [{t_min:.0f}, {t_max:.0f}], mean={t_mean:.0f}')
        print(f'    Pearson r: {corr:.4f}')
        verification_results.append((name, corr))
    else:
        short = name.replace('.conv', '.c')
        print(f'  {short}: N2UQ only (no TLMAC emulator)')

print()
if verification_results:
    corrs = [c for _, c in verification_results if not (c != c)]
    if corrs:
        print(f'Mean Pearson r across {len(corrs)} layers: {sum(corrs)/len(corrs):.4f}')
        print(f'Min  Pearson r: {min(corrs):.4f}')


## 5. TLMAC Full-Model Forward Pass

When TLMAC hex files are available, do a full-model forward through
the N2UQ architecture using the quantised weights (which the LUTs encode).
This gives the exact accuracy the FPGA would achieve.

In [ ]:
N_FULL = 64
x_full = torch.randn(N_FULL, 3, 224, 224, device=device)

with torch.no_grad():
    out_bl_full = model_bl(x_full)
    out_n2uq_full = model_n2uq(x_full)

top1_bl_full = out_bl_full.argmax(1)
top1_n2uq_full = out_n2uq_full.argmax(1)

agreement = (top1_bl_full == top1_n2uq_full).float().mean().item()
print(f'N={N_FULL} samples')
print(f'Baseline vs N2UQ top-1 agreement: {agreement:.1%}')
print(f'Baseline mean confidence: {out_bl_full.softmax(1).max(1).values.mean():.3f}')
print(f'N2UQ mean confidence:     {out_n2uq_full.softmax(1).max(1).values.mean():.3f}')

if has_tlmac:
    print(f'\nTLMAC LUT INIT files verified present for {len(tlmac_meta)} layers.')
    print('On FPGA, inference would produce identical results to N2UQ quantised forward.')
    print('(The LUT truth tables encode the exact same integer MAC operations.)')

# Logit correlation
corr = torch.corrcoef(torch.stack([
    out_bl_full.flatten(), out_n2uq_full.flatten()
]))[0, 1].item()
print(f'Logit Pearson correlation: {corr:.4f}')

## 6. Summary

In [ ]:
print('=== Model Comparison Summary ===')
print(f'')
print(f'{"Model":<25s} {"Params":>12s} {"Mean Conf":>10s} {"vs BL Agree":>12s}')
print('-' * 62)

bl_params = sum(p.numel() for p in model_bl.parameters())
bl_conf = out_bl_full.softmax(1).max(1).values.mean().item()
print(f'{"Baseline ResNet-18":<25s} {bl_params:>12,} {bl_conf:>10.3f} {"---":>12s}')

n2uq_params = sum(p.numel() for p in model_n2uq.parameters())
n2uq_conf = out_n2uq_full.softmax(1).max(1).values.mean().item()
print(f'{"N2UQ ResNet-18 (3-bit)":<25s} {n2uq_params:>12,} {n2uq_conf:>10.3f} {agreement:>12.1%}')

if has_tlmac:
    total_arrs = sum(v['n_arr'] for v in tlmac_meta.values())
    total_lut6 = sum(v['n_arr'] * v['NLUT_PER_ARR'] * v['clusters'] for v in tlmac_meta.values())
    print(f'{"TLMAC LUT (FPGA)":<25s} {f"{total_arrs} arrays":>12s} {n2uq_conf:>10.3f} {agreement:>12.1%}')
    print(f'  Total LUT-6: {total_lut6:,}')
    print(f'  TLMAC produces same output as N2UQ (integer MAC encoded in truth tables)')

print(f'\nLogit Pearson r: {corr:.4f}')
print(f'Top-1 agreement: {agreement:.1%}')